# 1 - Config

In [74]:
# 🔧 Config import
import logging
import sys
import os

try:
    current_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    current_dir = os.getcwd()

project_root = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(project_root)

from Config.LoggerConfig import colored_logger

logger = colored_logger()
current_file = os.path.basename(__file__) if '__file__' in globals() else "Notebook"
logger.info(f"Logger initialized ({current_file})")

print("\033[92mThis should be light green\033[0m")


[2025-06-01 14:52:23] INFO - 1801737798.py - Logger initialized (Notebook)


This should be light green


# 2 Data

## 2.1 - Data fetcher

In [77]:
from Data.Data_Fetcher import DataFetcher
import pandas as pd
import plotly.graph_objects as go

 #📊 Data parameters
symbol = "BTCUSDT"
start_date = "01-01-2025"
end_date = "24-04-2025"
interval = "5m"

In [78]:
# 📥 Load data

fetcher = DataFetcher(symbol, start_date, end_date, interval)

fetcher.get_binance_data()

fetcher.save_to_csv(directory="./Data")

fetcher.load_from_csv(directory="./Data")

[2025-06-01 14:55:07] INFO - Data_Fetcher.py - Fetching 5m Binance Futures data for BTCUSDT
[2025-06-01 14:55:45] INFO - Data_Fetcher.py - ✅ Data fetch completed: 32557 rows.
[2025-06-01 14:55:46] INFO - Data_Fetcher.py - 📁 Data saved to: ./Data/BTCUSDT_01-01-2025_24-04-2025_5m.csv
[2025-06-01 14:55:46] INFO - Data_Fetcher.py - 📥 Data loaded from: ./Data/BTCUSDT_01-01-2025_24-04-2025_5m.csv


In [79]:
print("#-----------------------")
print(fetcher.raw_data.columns)
print("#-----------------------")
print(fetcher.raw_data.head())
print("#-----------------------")
print(fetcher.raw_data.shape)
print("#-----------------------")

# Suppose fetcher.raw_data is a DataFrame with ['time', 'open', 'high', 'low', 'close']
df = fetcher.raw_data.copy()[-200:]

# Create candlestick chart
fig = go.Figure(data=[ go.Candlestick( x=df['Timestamp'], open=df['Open'], high=df['High'], low=df['Low'], close=df['Close'], name="Price" ) ])

# Update layout for better look
fig.update_layout( title="Market Price (Last 200 Points)", xaxis_title="Time", yaxis_title="Price", xaxis_rangeslider_visible=False, template="plotly_dark", height=400, width=400*1.618 )

# Show the figure in Jupyter
fig.show()

#-----------------------
Index(['timestamp', 'open', 'high', 'low', 'close', 'volume'], dtype='object')
#-----------------------
             timestamp     open     high      low    close   volume
0  2024-12-31 23:00:00  93469.1  93550.9  93362.9  93394.1  484.937
1  2024-12-31 23:05:00  93394.1  93623.1  93356.6  93623.1  433.904
2  2024-12-31 23:10:00  93623.0  93736.9  93576.0  93687.3  268.385
3  2024-12-31 23:15:00  93687.2  93689.7  93617.7  93631.1  140.948
4  2024-12-31 23:20:00  93631.1  93635.2  93563.6  93607.8  151.778
#-----------------------
(32557, 6)
#-----------------------


KeyError: 'Timestamp'

## 2.2 - Feature Data

In [ ]:
from Data.Features import LSTM_Features
import pandas as pd
import matplotlib.pyplot as plt
import mplfinance as mpf
# Step 1: Market Sessions
resample_period = '5min'

# Step 3: Is Noise
noise_ratio = 0.15

# Step 4: Pivot Points
pivot_left = 15
pivot_right = 15

# Step 5: Volume Pivot Points
duration_min = 240
n_cross = 7
std_factor = 1.0

In [ ]:
# Step 0: Initialize Features Dataframe
Features_Data = LSTM_Features(fetcher.raw_data)

#-------------------------------------------------------------------------------

print("#----------------------------------------------------------------------")
print("Shape:", Features_Data.data.shape)
print("#----------------------------------------------------------------------")
print(Features_Data.data.columns)
print("#----------------------------------------------------------------------")
print(Features_Data.data.head())
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------
# Step 1: Market Sessions
Features_Data.resample_with_vwap(resample_period)

# Step 2: Market Sessions
Features_Data.market_sessions()

# Step 3: Is Noise
Features_Data.is_noise(noise_ratio)

# Step 4: Pivot Points
Features_Data.Pivot_Points(pivot_left, pivot_right)

# Step 5: Volume Pivot Points
Features_Data.Volume_Pivot_Points(duration_min, n_cross, std_factor)

In [ ]:
#-------------------------------------------------------------------------------

print("#----------------------------------------------------------------------")
print("Shape:", Features_Data.data.shape)
print("#----------------------------------------------------------------------")
print(Features_Data.data.columns)
print("#----------------------------------------------------------------------")
print(Features_Data.data.head())
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------


apds = [mpf.make_addplot(Features_Data.data.tail(20)['VWAP_5m'], color='blue', panel=0)]
mpf.plot(Features_Data.data.tail(20), type='candle', style='yahoo', title='Candlestick Chart with VWAP_5m',ylabel='Price', addplot=apds, volume=False)
plt.show()
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------

frequency = Features_Data.data[["London_Open", "NY_Open", "HK_Open"]].apply(lambda col: pd.Series(col).value_counts()).fillna(0).astype(int)
print("\nSession Open/Close Frequency (Top 3):")
print(frequency.head(3))
print("#----------------------------------------------------------------------")
plt.figure(figsize=(12, 4))
plt.plot(Features_Data.data["London_Open"], label="London")
plt.plot(Features_Data.data["NY_Open"], label="New York")
plt.plot(Features_Data.data["HK_Open"], label="Hong Kong")
plt.legend()
plt.title(f"Market Sessions")
plt.xlabel("Time")
plt.ylabel("Market Open (1 = open, 0 = closed)")
plt.show()
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------

frequency = Features_Data.data["Is_Noise"].value_counts().sort_index()
print("\nNoise Label Frequency:")
print(frequency)
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------

plt.figure(figsize=(12, 6))
plt.plot(Features_Data.data["Low"], label="Low", linewidth=0.8)
plt.plot(Features_Data.data["Low_Pivot"], label="Low Pivot", linewidth=1.2, linestyle='--')
plt.plot(Features_Data.data["High"], label="High", linewidth=0.8)
plt.plot(Features_Data.data["High_Pivot"], label="High Pivot", linewidth=1.2, linestyle='--')
plt.legend()
plt.title("Pivot Points Detection (5min Resampled Data)")
plt.xlabel("Time")
plt.ylabel("Price")
plt.grid(True)
plt.tight_layout()
plt.show()
print("#----------------------------------------------------------------------")

#-------------------------------------------------------------------------------

plt.figure(figsize=(12, 6))
plt.plot(Features_Data.data["Volume_Low_Pivot"] , label="Volume Low Pivot", linewidth=1.2, linestyle='--')
plt.plot(Features_Data.data["VWAP_5m"], label="VWAP 5m", linewidth=0.8)
plt.plot(Features_Data.data['Rolling_VWAP_240min'])
plt.plot(Features_Data.data["Volume_High_Pivot"] , label="Volume High Pivot", linewidth=1.2, linestyle='--')
plt.legend()
plt.title("Volume Pivot Points Detection (5min Resampled Data)")
plt.xlabel("Time")
plt.ylabel("Price")
plt.grid(True)
plt.tight_layout()
plt.show()
print("#----------------------------------------------------------------------")

## 2.3 - Label Data

## 2.4 - Data Analysis